# Student Performance: Final Integrated Case

**Source:** UCI Machine Learning Repository, Student Performance dataset (student-mat.csv and student-por.csv).
**Acquisition:** Extracted from student+performance.zip (it contained a second zip, student.zip). Files are semicolon-separated.
**Raw data policy:** The raw files are never edited. All cleaning is done on copies.

In [1]:
import pandas as pd

math = pd.read_csv("student_data/student-mat.csv", sep=";")
por = pd.read_csv("student_data/student-por.csv", sep=";")

print(math.shape, por.shape)

(395, 33) (649, 33)


## Grain and key statement

**Grain:** One row is one student's record for one course. A student who took both courses appears once in each file.

**Identifiers vs measures:** There is no student ID. Columns like school, sex, age, address, famsize, Pstatus, Medu, Fedu, Mjob, Fjob, reason, nursery and internet describe who the student is, so they are used as a composite key to match students across files. Grades (G1, G2, G3) are measures and are never used for matching.

**Key assumption:** These 13 columns are treated as a student's identity. They may not be unique, so uniqueness is checked before merging.

In [2]:
health = pd.DataFrame({
    "dataset": ["Math", "Portuguese"],
    "rows": [len(math), len(por)],
    "columns": [math.shape[1], por.shape[1]],
    "missing_values": [math.isna().sum().sum(), por.isna().sum().sum()],
    "duplicate_rows": [math.duplicated().sum(), por.duplicated().sum()],
    "G3_min": [math["G3"].min(), por["G3"].min()],
    "G3_max": [math["G3"].max(), por["G3"].max()],
})

health

,dataset,rows,columns,missing_values,duplicate_rows,G3_min,G3_max
0,Math,395,33,0,0,0,20
1,Portuguese,649,33,0,0,0,19


In [3]:
keys = ["school", "sex", "age", "address", "famsize", "Pstatus", "Medu", "Fedu",
        "Mjob", "Fjob", "reason", "nursery", "internet"]

support = math.loc[
    (math["G3"] < 10) | (math["failures"] > 0) | (math["absences"] >= 10)
].copy()

math_u = math[~math.duplicated(keys, keep=False)].copy()
por_u = por[~por.duplicated(keys, keep=False)].copy()

population = pd.DataFrame({
    "population": ["All Math students", "Support group (Math)",
                   "Math, unique keys", "Portuguese, unique keys"],
    "rows": [len(math), len(support), len(math_u), len(por_u)],
})

population

,population,rows
0,All Math students,395
1,Support group (Math),195
2,"Math, unique keys",387
3,"Portuguese, unique keys",626


In [4]:
m = math.copy()

text_cols = ["school", "sex", "address", "famsize", "Pstatus", "Mjob", "Fjob",
             "reason", "guardian", "schoolsup", "famsup", "paid", "activities",
             "nursery", "higher", "internet", "romantic"]

for c in text_cols:
    m[c] = m[c].astype("string").str.strip()

print(m.shape)
print((m[text_cols].astype("object") == math[text_cols].astype("object")).all().all())

(395, 33)
True


## Cleaning decision log

| Decision | Why | Effect |
|---|---|---|
| Worked on copies (m, math_u, por_u), never on the raw files | Keeps the raw data reproducible | math and por unchanged |
| Stripped spaces from 17 text columns | Prevents hidden-space mismatches in grouping and merging | No values changed, so the text was already clean |
| No rows dropped for missing values | Health check found 0 missing values in both files | Nothing to drop or impute |
| No exact duplicate rows removed | Health check found 0 duplicate rows | Row counts unchanged |
| Kept students with G3 = 0 | A zero may mean the exam was not taken, but the data cannot confirm that, so removing them would change the meaning | Flagged as a limitation, not deleted |
| Excluded students sharing identical key values (8 Math, 23 Portuguese) | They cannot be matched reliably across files | Applies to the merge only |

In [5]:
m["passed"] = m["G3"].ge(10)
m["grade_change"] = m["G3"] - m["G1"]
m["absence_band"] = pd.cut(
    m["absences"],
    [-1, 5, 10, 20, float("inf")],
    labels=["low", "moderate", "high", "very high"]
)

print(m.shape)
print(m["passed"].value_counts())
print(m["absence_band"].value_counts(dropna=False))

(395, 36)
passed
True     265
False    130
Name: count, dtype: int64
absence_band
low          249
moderate      80
high          51
very high     15
Name: count, dtype: int64


In [6]:
summary1 = (
    m.groupby(["school", "studytime"], as_index=False)
    .agg(students=("G3", "size"),
         mean_final=("G3", "mean"),
         pass_rate=("passed", "mean"))
)
summary1["pass_rate_pct"] = summary1["pass_rate"] * 100

assert summary1["students"].sum() == len(m)

print(summary1.shape)
summary1.sort_values("mean_final", ascending=False)


(7, 6)


,school,studytime,students,mean_final,pass_rate,pass_rate_pct
6,MS,3,8,11.750000,0.875000,87.500000
2,GP,3,57,11.350877,0.736842,73.684211
3,GP,4,27,11.259259,0.740741,74.074074
5,MS,2,22,10.590909,0.727273,72.727273
0,GP,1,89,10.438202,0.696629,69.662921
1,GP,2,176,10.119318,0.636364,63.636364
4,MS,1,16,7.875000,0.375000,37.500000


In [7]:
summary2 = (
    m.groupby("absence_band", observed=True, as_index=False)
    .agg(students=("G3", "size"),
         mean_final=("G3", "mean"),
         pass_rate=("passed", "mean"))
)

assert summary2["students"].sum() == len(m)

print(summary2.shape)
summary2

(4, 4)


,absence_band,students,mean_final,pass_rate
0,low,249,10.168675,0.682731
1,moderate,80,11.400000,0.737500
2,high,51,10.117647,0.568627
3,very high,15,10.266667,0.466667


In [8]:
both = math_u.merge(
    por_u,
    on=keys,
    how="inner",
    suffixes=("_math", "_por"),
    validate="one_to_one",
    indicator=True
)
both["final_gap"] = both["G3_math"] - both["G3_por"]

print(both.shape)
print(both["_merge"].value_counts())
print(both["final_gap"].mean())

(358, 55)
_merge
both          358
left_only       0
right_only      0
Name: count, dtype: int64
-2.036312849162011


In [9]:
# 1. Cleaning added columns but did not lose or add rows
assert len(m) == len(math) == 395

# 2. Merge keys are unique in both cleaned tables
assert not math_u.duplicated(keys).any()
assert not por_u.duplicated(keys).any()

# 3. Merge cannot produce more students than the smaller table
assert len(both) <= min(len(math_u), len(por_u))

print("All validation checks passed")

All validation checks passed


In [10]:
import os
os.makedirs("outputs", exist_ok=True)

m.to_csv("outputs/math_clean.csv", index=False)
both.to_csv("outputs/math_por_merged.csv", index=False)
summary1.to_csv("outputs/summary_school_studytime.csv", index=False)

print(os.listdir("outputs"))
print(m.shape, both.shape, summary1.shape)

['math_clean.csv', 'math_por_merged.csv', 'summary_school_studytime.csv']
(395, 36) (358, 55) (7, 6)


## Management note

This analysis uses the UCI Student Performance data for two courses: Math (395 students) and Portuguese (649 students). Both files were complete, with no missing values or duplicate rows, and all grades were within the 0 to 20 range.

**Findings.** 265 of 395 Math students passed (final grade of 10 or more). At school GP, students who study more did better: pass rates were about 74% for study time codes 3 and 4, against about 64% to 70% for codes 1 and 2. Pass rates were highest for students with moderate absences (74%) and fell to 57% for high absences and 47% for very high absences. Among the 358 students found in both courses, Math grades averaged about 2 points lower than Portuguese (10.55 vs 12.58).

**Exclusions.** There is no student ID, so students were matched on 13 demographic and school fields, never on grades. Students sharing identical values on those fields (8 in Math, 23 in Portuguese) were excluded from the merge because they cannot be matched reliably.

**Assumptions and limitations.** 38 Math students have a final grade of 0 and no absences, which may mean an exam was not taken. They were kept and flagged, not deleted. Some groups are very small, such as school MS with study time 3 (8 students). These results show associations, not causes, and the two courses may be graded differently, so a grade gap does not show that students are weaker at Math. Excluding the lookalike students may also leave the matched group slightly unrepresentative.